# Confirmed-state access latency in a permissioned Avalanche L1
## Затримка доступу до підтвердженого стану в дозволеній мережі Avalanche L1

**Reproducible field experiment — end to end, in this notebook.**

This notebook runs the complete pipeline of the study: the randomized run
schedule, the immutable workload traces, a dataset in the campaign's exact
schema, the full statistical plan, every table and figure of the paper, the
SHA-256 manifests, a byte-for-byte reproduction check, and the archive for
the Zenodo deposition.

**Runtime:** free Colab CPU. **Wall clock:** about one minute for the `demo`
profile. **Nothing to configure.** Run all cells.

---

### Провенанс / Provenance — read this first

The provided sources confirm the infrastructure and the design but contain
**no raw logs and no measured p50/p95/p99**. So this notebook generates its
dataset from a *documented reference model* (`alp.simulate`). Every record,
table and figure it produces is labelled `SIMULATED`.

> **Ці числа не є вимірюваннями кіберполігона і не можуть подаватися як
> результати експерименту.** They exist to prove the analysis pipeline is
> complete, correct and reproducible before the real campaign runs.

Section 9 shows how to point the identical pipeline at **real campaign
logs**: nothing downstream changes.

## 1. Setup

Two ways to get the project into Colab. The first cell tries `git clone`;
if the repository is private or unreachable, use the upload fallback below
it (upload the Zenodo `.zip` and it will be unpacked).

In [ ]:
# @title Get the project (clone) { display-mode: "form" }
REPO_URL = "https://github.com/omega2417/bnt.git"  # @param {type:"string"}
BRANCH = "claude/reproducible-field-experiment-zenodo-37ds4v"  # @param {type:"string"}
SUBDIR = "avalanche-latency-experiment"  # @param {type:"string"}

import os, pathlib, subprocess, sys

PROJECT = pathlib.Path.cwd()
if not (PROJECT / "src" / "alp").is_dir():
    target = pathlib.Path("/content/bnt")
    if not target.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(target)],
            check=False,
        )
    candidate = target / SUBDIR
    if candidate.is_dir():
        PROJECT = candidate

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))
# The `!python -m alp ...` cells below run in subprocesses, which do not
# inherit sys.path. Exporting PYTHONPATH is what makes them find the package.
os.environ["PYTHONPATH"] = str(PROJECT / "src")

print("project directory:", PROJECT)
print("package present:", (PROJECT / "src" / "alp" / "cli.py").exists())

In [ ]:
# Fallback: upload the Zenodo archive instead of cloning.
# Run this cell ONLY if the clone above did not find the package.
#
# from google.colab import files
# import zipfile, pathlib, os, sys
# uploaded = files.upload()                      # choose avalanche-latency-experiment*.zip
# name = next(iter(uploaded))
# with zipfile.ZipFile(name) as zf:
#     zf.extractall("/content/unpacked")
# root = next(p for p in pathlib.Path("/content/unpacked").iterdir() if p.is_dir())
# os.chdir(root); sys.path.insert(0, str(root / "src"))
# os.environ["PYTHONPATH"] = str(root / "src")
# print("project directory:", root)

In [ ]:
# Dependencies. Colab already ships numpy/pandas/matplotlib/scipy, so this is
# usually a no-op; it is here so the notebook also runs on a bare kernel.
%pip install -q -r requirements.txt

import platform, sys
import matplotlib, numpy, pandas, scipy
import alp

print(f"python      {sys.version.split()[0]}   ({platform.platform()})")
print(f"alp         {alp.__version__}")
print(f"numpy       {numpy.__version__}")
print(f"pandas      {pandas.__version__}")
print(f"scipy       {scipy.__version__}")
print(f"matplotlib  {matplotlib.__version__}")

## 2. What the protocol commits to, before any data exists

Equations (1)–(4) size the campaign; equations (15)–(16) and the uniform
block-phase argument give the closed-form expectations. These are `THEORY`
and `DERIVED` values — the protocol forbids copying them into a Results
table, and the pipeline keeps them in separate files for exactly that reason.

In [ ]:
from alp import config, theory

FULL = config.FULL
print(f"pre-registered campaign : {FULL.n_runs} runs")
print(f"scheduled transactions  : {FULL.n_scheduled_tx:,}")
print(f"minimum machine time    : {FULL.wall_clock_s/3600:.2f} h")
print(f"run structure           : {FULL.warmup_s} s warm-up + "
      f"{FULL.measure_s} s measurement + {FULL.drain_s} s drain")
print()
display(theory.campaign_arithmetic(FULL))

In [ ]:
# Tab. 9 and Tab. 10: nominal block rate, and the block-wait component alone
# under W ~ U(0, B).  Not T_visible — the structural lower bound of it.
display(theory.table_nominal_rate())
display(theory.table_block_wait())

## 3. The design: randomized schedule and immutable traces

Configuration order is randomised inside every `topology × load` block, so
time of day, ambient campus traffic and thermal drift cannot line up with a
configuration. Within a `topology × load × repeat` stratum every
configuration replays **the same** pre-computed arrival trace — that is what
makes the design paired and the bootstrap valid.

In [ ]:
from alp.schedule import build_schedule
from alp.traces import build_all_traces

schedule = build_schedule("full")
print(f"schedule: {len(schedule)} runs, {schedule.run_id.nunique()} unique ids")
display(schedule.head(8))

# One stratum: five configurations, one shared trace.
stratum = schedule[(schedule.topology == "T1_vpn") &
                   (schedule.load_tps == 100) &
                   (schedule.repeat == 1)]
print("\nSame trace under every configuration in one stratum:")
display(stratum[["run_id", "config", "trace_id", "order"]])

In [ ]:
registry = build_all_traces(config.DEMO)
print(f"{len(registry)} immutable traces, each hashed into the run passport")
display(registry.head())

## 4. Produce the dataset

`demo` keeps **every factor level** of the protocol (5 configurations × 3
topologies × 5 loads) and shortens the measurement window and the repeat
count so the campaign fits a notebook session. Switch `PROFILE` to `full`
to run the pre-registered 750-run campaign — it takes a few minutes and
produces a few GB.

What the model represents, and what it deliberately does not, is documented
in `protocol/MODEL.md`.

In [ ]:
PROFILE = "demo"   # "smoke" (fastest) | "demo" (default) | "full" (750 runs)

profile = config.get_profile(PROFILE)
print(f"{profile.name}: {profile.n_runs} runs, "
      f"{profile.n_scheduled_tx:,} scheduled transactions, "
      f"{profile.measure_s} s measurement window")

!python -m alp simulate --profile {PROFILE} --clean

In [ ]:
# One raw record, exactly as a load generator writes it in the field.
import gzip, json, pathlib

sample = sorted(pathlib.Path("data/raw/tx").glob("*.jsonl.gz"))[0]
with gzip.open(sample, "rt") as fh:
    record = json.loads(fh.readline())
print(sample.name)
print(json.dumps(record, indent=2))

## 5. The statistical plan

Two commitments drive everything here.

**The run is the inferential unit.** Transactions inside a run share a
block, a consensus round and a disk queue, so they are not independent
replicates. Bootstrapping individual transactions would produce
artificially narrow intervals. Every metric is reduced to run level first.

**The comparison is paired.** Inside each `topology × load` stratum the
baseline C0 and the profile share a trace, so the difference is taken run by
run before resampling.

In [ ]:
!python -m alp analyze --profile {PROFILE}

In [ ]:
import pandas as pd
from alp import analyze

summary = pd.read_csv("results/run_level_summary.csv")
print("run-level summary — the inferential unit")
display(summary.head())

print(f"\ndataset provenance: {analyze.dataset_provenance('data/raw')}")

## 6. Results tables

Tab. 14 (latency quantiles), Tab. 15 (stability, goodput, resources) and
Tab. 16 (paired effects against C0 with 95 % confidence intervals). Each is
also written as CSV, Markdown and JTIT-style LaTeX under `results/`.

In [ ]:
t14 = pd.read_csv("results/csv/table14_latency_quantiles.csv")
t15 = pd.read_csv("results/csv/table15_stability.csv")
t16 = pd.read_csv("results/csv/table16_effects.csv")
reach = pd.read_csv("results/csv/table_max_sustainable_load.csv")
best = pd.read_csv("results/csv/table_best_static.csv")

display(t14.head(15))
display(t15.head(15))
display(t16.head(15))
display(reach)
display(best)

## 7. Figures

Figures 1–4 are schematics and closed-form curves that exist before any
data. Figures 5–11 are result figures and carry the provenance of the
dataset they were built from in a footer.

In [ ]:
from IPython.display import Image, display as show
import pathlib

for path in sorted(pathlib.Path("results/figures").glob("*.png")):
    print(path.name)
    show(Image(filename=str(path), width=900))

## 8. The generated report

`results/RESULTS.md` and `results/RESULTS_UK.md` are written by code: every
sentence containing a number pulls it from the analysis outputs, so the text
cannot drift away from the data. The hypothesis verdicts follow decision
rules stated in `alp.report.hypothesis_verdicts`, not a reading of the
tables.

In [ ]:
from IPython.display import Markdown

Markdown(pathlib.Path("results/RESULTS.md").read_text(encoding="utf-8"))

In [ ]:
# Українською
Markdown(pathlib.Path("results/RESULTS_UK.md").read_text(encoding="utf-8"))

## 9. Reproducibility, demonstrated rather than claimed

Three checks:

1. **Manifests** — re-hash every artefact and compare to `MANIFEST.sha256`.
2. **Re-derivation** — regenerate the entire dataset from the master seed
   into a scratch tree and diff every derived table against the committed one.
3. **Tests** — the protocol arithmetic, each metric equation, the design
   invariants and the end-to-end pipeline.

In [ ]:
!python -m alp manifest data/raw results --profile {PROFILE}
!python -m alp verify data/raw results

In [ ]:
# Regenerate everything from the seed and diff the derived tables.
!python -m alp reproduce --profile smoke --results results

In [ ]:
%pip install -q pytest
!python -m pytest tests -q

## 10. Bring your own logs — the real campaign

When the cyber-range campaign produces raw logs, nothing in the analysis
changes. Place the JSONL under `data/raw/tx/` (with the node and network
files beside them, per `protocol/DATA_DICTIONARY.md`) and run the pipeline
with `--skip-simulation`.

The provenance label follows the records, so every table, figure and report
will say `MEASURED` instead of `SIMULATED`. Before publishing any number,
complete `protocol/DATA_REQUIRED.md`.

In [ ]:
# Uncomment when real logs are present under data/raw/tx/
#
# !python -m alp pipeline --profile full --skip-simulation \
#     --data data/raw --results results --provenance MEASURED
#
# Field-side entry points, for reference:
#   deploy/preflight.sh              isolation, disk, clock and node health gates
#   deploy/run_one.sh                one run: passport, profile, netem, collect, hash
#   python -m alp.client trace.csv   the measurement client on each workstation
print(open("protocol/DATA_REQUIRED.md", encoding="utf-8").read()[:1500])

## 11. Build the Zenodo archive

The archive is built deterministically — sorted entries, pinned timestamps,
fixed compression — so the same tree always produces the same SHA-256. That
matters because Zenodo mints a DOI for the exact bytes it receives.

Deposition metadata is in `.zenodo.json`; citation metadata in
`CITATION.cff`. Step-by-step instructions: `docs/ZENODO.md`.

In [ ]:
!python -m alp package --out dist/avalanche-latency-experiment.zip
!cat dist/avalanche-latency-experiment.zip.sha256

In [ ]:
# Download the archive from Colab.
try:
    from google.colab import files
    files.download("dist/avalanche-latency-experiment.zip")
except ImportError:
    print("not running in Colab; the archive is at "
          "dist/avalanche-latency-experiment.zip")

---

### What to read next

| File | What it holds |
| --- | --- |
| `protocol/PROTOCOL.md` · `protocol/PROTOCOL_UK.md` | the executable specification |
| `protocol/MODEL.md` | what the reference model does and does not represent |
| `protocol/DATA_DICTIONARY.md` | every field of every file |
| `protocol/DATA_REQUIRED.md` | what a real campaign must supply |
| `docs/REPRODUCE.md` | reproduction outside Colab |
| `docs/ZENODO.md` | deposition checklist |
| `results/article_fragment.tex` | the Experimental Setup and Results paragraphs, with numbers |